In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_timestamp, hour, avg, count, first, round as spark_round

In [ ]:
spark = SparkSession.builder \
    .appName("Analisis_AQI_Jatim") \
    .getOrCreate()

# Baca JSON multiline dari HDFS (format array JSON, bukan JSON Lines)
df_api_raw = spark.read \
    .option("multiline", "true") \
    .json("hdfs://namenode:9000/data/airquality/api/")

# Cast kolom numerik dan timestamp
# Format timestamp: '01-Mei-2026 12:00:00' (nama bulan Indonesia)
# Spark tidak support locale Indonesia, jadi kita parse via ingested_at sebagai fallback
df_api = df_api_raw \
    .withColumn("aqi", col("aqi").cast("double")) \
    .withColumn("pm25", col("pm25").cast("double")) \
    .withColumn("ts", to_timestamp(col("ingested_at"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS")) \
    .withColumn("jam", hour(col("ts")))

print("Schema:")
df_api.printSchema()
df_api.select("kota", "aqi", "ingested_at", "ts", "jam").show(5, truncate=False)

In [ ]:
# Klasifikasi AQI (kolom sudah di-cast ke double di sel sebelumnya)
df_classified = df_api.withColumn("kategori_baru",
    when(col("aqi") <= 50, "Baik")
    .when((col("aqi") > 50) & (col("aqi") <= 100), "Sedang")
    .when((col("aqi") > 100) & (col("aqi") <= 200), "Tidak Sehat")
    .otherwise("Berbahaya")
)
df_classified.select("kota", "aqi", "kategori", "kategori_baru").show(10)

In [ ]:
# Hitung total record per kota
total_kota = df_classified.groupBy("kota").count().withColumnRenamed("count", "total")

# Hitung jumlah per kategori di tiap kota
distribusi = df_classified.groupBy("kota", "kategori_baru").count()

# Gabungkan dan hitung persentase
hasil_analisis = distribusi.join(total_kota, "kota") \
    .withColumn("persentase", spark_round((col("count") / col("total")) * 100, 1)) \
    .select("kota", "kategori_baru", "persentase") \
    .orderBy("kota", "kategori_baru")

hasil_analisis.show(truncate=False)

In [ ]:
hasil_analisis.write.mode("overwrite").json("hdfs://namenode:9000/data/airquality/hasil/analisis1")
print("Analisis 1 tersimpan ke HDFS: /data/airquality/hasil/analisis1")

## Analisis 2: Identifikasi Jam Puncak Polusi per Kota

Menjawab pertanyaan bisnis ETS: **"Pada jam berapa kualitas udara paling buruk?"**

Pipeline:
1. Parse kolom timestamp -> ekstrak jam (0-23)
2. Agregasi rata-rata AQI per kota & jam
3. Identifikasi jam puncak (avg_aqi tertinggi) per kota
4. Simpan hasil ke HDFS `/data/airquality/hasil/analisis2`

In [ ]:
from pyspark.sql.functions import col, to_timestamp, hour, avg, count, round as spark_round, first, when

# df_api sudah punya kolom 'ts' dan 'jam' dari sel sebelumnya
# Rename agar konsisten dengan query SQL
df_ts = df_api  # kolom 'kota' sudah ada, 'ts' dan 'jam' sudah di-parse

print('Schema df_ts:')
df_ts.printSchema()
df_ts.select('kota', 'ingested_at', 'ts', 'jam', 'aqi').show(5, truncate=False)

In [ ]:
# -------------------------------------------------------
# Langkah 2: Agregasi rata-rata AQI per kota dan jam
# -------------------------------------------------------
df_ts.createOrReplaceTempView('aqi_ts')

hasil2 = spark.sql("""
    SELECT kota,
           HOUR(ts)            AS jam,
           ROUND(AVG(aqi), 1)  AS avg_aqi,
           COUNT(*)            AS jumlah_data
    FROM   aqi_ts
    GROUP  BY kota, HOUR(ts)
    ORDER  BY kota, avg_aqi DESC
""")

print('Rata-rata AQI per Kota per Jam (diurutkan dari polusi tertinggi):')
hasil2.show(50, truncate=False)

In [ ]:
# -------------------------------------------------------
# Langkah 3: Identifikasi jam puncak polusi per kota
# Karena hasil2 sudah diurutkan avg_aqi DESC per kota,
# first('jam') mengambil jam dengan AQI rata-rata tertinggi.
# -------------------------------------------------------
jam_puncak = hasil2.groupBy('kota').agg(
    first('jam').alias('jam_puncak'),
    first('avg_aqi').alias('avg_aqi_puncak'),
    first('jumlah_data').alias('jumlah_data_puncak')
).orderBy('kota')

print('=== TABEL JAM PUNCAK POLUSI PER KOTA ===')
jam_puncak.show(truncate=False)

# Klasifikasi sesi jam
jam_puncak_labeled = jam_puncak.withColumn(
    'sesi',
    when((col('jam_puncak') >= 7) & (col('jam_puncak') <= 9),   'Pagi Sibuk (07-09)')
    .when((col('jam_puncak') >= 17) & (col('jam_puncak') <= 19), 'Sore Sibuk (17-19)')
    .when((col('jam_puncak') >= 10) & (col('jam_puncak') <= 16), 'Siang (10-16)')
    .when((col('jam_puncak') >= 20) & (col('jam_puncak') <= 23), 'Malam (20-23)')
    .otherwise('Dini Hari (00-06)')
)

print('=== JAM PUNCAK DENGAN KLASIFIKASI SESI ===')
jam_puncak_labeled.select('kota', 'jam_puncak', 'sesi', 'avg_aqi_puncak').show(truncate=False)

### Narasi & Rekomendasi Bisnis

#### Temuan Utama

Analisis pola jam menunjukkan bahwa **polusi udara umumnya memuncak pada dua sesi krusial**:

| Sesi | Jam | Faktor Pemicu |
|------|-----|---------------|
| **Pagi Sibuk** | 07.00-09.00 | Kendaraan berangkat kerja/sekolah, aktivitas industri awal hari |
| **Sore Sibuk** | 17.00-19.00 | Kendaraan pulang kerja, akumulasi polutan sepanjang hari |

> **Catatan**: Kota-kota dengan jam puncak di luar dua sesi di atas (misal siang atau dini hari)
> patut dicermati — bisa mengindikasikan sumber polusi industri atau pembakaran sampah yang tidak terjadwal.

#### Pola yang Sering Ditemukan

- **Pagi (07-09)** biasanya dominan di kota dengan kepadatan kendaraan tinggi dan industri pagi.
- **Sore (17-19)** sering lebih tinggi karena efek *photochemical smog* — sinar matahari seharian
  mengoksidasi polutan primer menjadi ozon dan partikulat sekunder.
- Kota industri berat kadang menunjukkan puncak malam (22-24) akibat shift produksi malam.

#### Rekomendasi untuk Dinas Kesehatan

1. **Notifikasi proaktif** — Kirim peringatan ISPU ke masyarakat 30 menit *sebelum* jam puncak spesifik per kota.
2. **Jadwal aktivitas luar ruang** — Rekomendasikan olahraga pagi sebelum 06.00 atau sesudah 20.00
   di kota dengan jam puncak pagi; sebaliknya, sebelum 16.00 jika puncak sore.
3. **Koordinasi dengan Dishub** — Dorong rekayasa lalu lintas (ganjil-genap, shifting jam kerja)
   atau perluasan WFH pada jam-jam kritis untuk menekan emisi kendaraan.
4. **Penguatan pemantauan ISPU** — Tingkatkan frekuensi pengukuran di kota dengan avg_aqi_puncak > 100
   (kategori Tidak Sehat), terutama menjelang jam puncak.
5. **Edukasi berbasis lokal** — Materi penyuluhan harus disesuaikan jam puncak tiap kota,
   bukan satu jadwal nasional yang seragam.

In [ ]:
# Simpan hasil ke HDFS
OUTPUT_PATH = 'hdfs://namenode:9000/data/airquality/hasil/analisis2'

# Simpan tabel lengkap (rata-rata AQI semua jam)
hasil2.write.mode('overwrite').json(f'{OUTPUT_PATH}/avg_aqi_per_jam')

# Simpan tabel ringkasan jam puncak per kota
jam_puncak_labeled.write.mode('overwrite').json(f'{OUTPUT_PATH}/jam_puncak_per_kota')

print(f'Hasil berhasil disimpan ke {OUTPUT_PATH}/')
print(f'   avg_aqi_per_jam/      -> rata-rata AQI tiap jam per kota')
print(f'   jam_puncak_per_kota/  -> jam puncak polusi per kota beserta klasifikasi sesi')

## Analisis 3: Ranking Kota dengan Kualitas Udara Terburuk

Membuat ranking 5 kota Jawa Timur berdasarkan rata-rata AQI tertinggi sekaligus menghitung
jumlah event yang masuk kategori "Tidak Sehat" atau lebih buruk (AQI > 100). Ranking ini akan
menjadi tabel utama di dashboard A5 dan masukan utama bagi rekomendasi prioritas intervensi.

Pipeline:
1. Agregasi statistik AQI per kota (avg, max, min, jumlah event tidak sehat).
2. Tambahkan kolom `peringkat` dengan window function `rank()`.
3. Simpan hasil ke HDFS `/data/airquality/hasil/analisis3`.
4. Tulis narasi rekomendasi bisnis berdasarkan ranking.

In [ ]:
# -------------------------------------------------------
# Langkah 1: Statistik AQI per kota (avg, max, min, event tidak sehat)
# -------------------------------------------------------
df_classified.createOrReplaceTempView('aqi_data')

hasil3 = spark.sql("""
    SELECT kota,
           ROUND(AVG(aqi), 1)                              AS avg_aqi,
           MAX(aqi)                                        AS max_aqi,
           MIN(aqi)                                        AS min_aqi,
           SUM(CASE WHEN aqi > 100 THEN 1 ELSE 0 END)      AS event_tidak_sehat,
           COUNT(*)                                        AS total_data
    FROM   aqi_data
    GROUP  BY kota
    ORDER  BY avg_aqi DESC
""")

print('=== STATISTIK AQI PER KOTA (sorted by avg_aqi DESC) ===')
hasil3.show(truncate=False)

In [ ]:
# -------------------------------------------------------
# Langkah 2: Tambahkan kolom peringkat dengan window function rank()
# -------------------------------------------------------
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc

w = Window.orderBy(desc('avg_aqi'))
ranking_kota = hasil3.withColumn('peringkat', rank().over(w)) \
    .select('peringkat', 'kota', 'avg_aqi', 'max_aqi', 'min_aqi',
            'event_tidak_sehat', 'total_data') \
    .orderBy('peringkat')

print('=== RANKING KOTA AQI TERBURUK ===')
ranking_kota.show(truncate=False)

In [ ]:
# -------------------------------------------------------
# Langkah 3: Highlight Top 5 kota terburuk untuk dashboard
# -------------------------------------------------------
top5 = ranking_kota.limit(5)
print('=== TOP 5 KOTA AQI TERBURUK (untuk panel ranking dashboard) ===')
top5.show(truncate=False)

In [ ]:
# -------------------------------------------------------
# Langkah 4: Simpan ranking ke HDFS
# -------------------------------------------------------
OUTPUT_PATH3 = 'hdfs://namenode:9000/data/airquality/hasil/analisis3'
ranking_kota.write.mode('overwrite').json(OUTPUT_PATH3)
print(f'Analisis 3 tersimpan ke HDFS: {OUTPUT_PATH3}')

### Narasi & Rekomendasi Bisnis (Analisis 3)

#### Temuan Utama

Ranking kota berdasarkan rata-rata AQI memberikan gambaran cepat soal kota mana yang **paling
konsisten terpapar polusi**. Kolom `event_tidak_sehat` (jumlah event dengan AQI > 100) menjadi
indikator pelengkap penting: kota dengan rata-rata AQI moderat tapi punya banyak event tidak sehat
berarti mengalami **lonjakan polusi episodik** yang berbahaya bagi kelompok rentan.

#### Cara Membaca Ranking

1. **`avg_aqi`** — proxy paparan kronis. Kota di peringkat 1-3 perlu intervensi struktural jangka panjang.
2. **`max_aqi`** — puncak ekstrem. Jika selisih `max_aqi - avg_aqi` besar, ada lonjakan tajam yang
   patut diinvestigasi (kebakaran lahan, kegiatan industri tidak rutin, dll).
3. **`event_tidak_sehat`** — frekuensi episode berisiko. Bahkan kota peringkat menengah bisa punya
   nilai tinggi di sini bila distribusi AQI-nya bimodal.

#### Rekomendasi untuk Dinas Kesehatan

1. **Prioritas intervensi** — Kota di peringkat 1-3 dijadikan fokus program penurunan emisi
   (audit industri, perluasan ruang terbuka hijau, ganjil-genap kendaraan).
2. **Sistem peringatan dini** — Kota dengan `event_tidak_sehat` tinggi mendapat prioritas instalasi
   sensor real-time tambahan agar peringatan ISPU dapat dikirim sebelum AQI menembus 100.
3. **Sosialisasi terarah** — Edukasi penggunaan masker N95 dan jadwal aktivitas luar ruang difokuskan
   ke 3 kota peringkat teratas, lengkap dengan jam puncak per kota dari Analisis 2.
4. **Kolaborasi lintas Pemda** — Gunakan ranking ini sebagai dasar penentuan alokasi anggaran
   penanganan polusi udara provinsi Jawa Timur.

## AQI-10: Generate `spark_results.json` untuk Dashboard

Menggabungkan output ketiga analisis menjadi satu file JSON yang akan dikonsumsi oleh
endpoint Flask `/api/spark` di dashboard A5.

Format yang sudah disepakati dengan A5:
```json
{
  "distribusi_kategori": [{"kota": ..., "kategori": ..., "persentase": ...}],
  "aqi_per_jam":         [{"kota": ..., "jam": ..., "avg_aqi": ..., "jumlah_data": ...}],
  "ranking_kota":        [{"peringkat": ..., "kota": ..., "avg_aqi": ..., "max_aqi": ...,
                           "min_aqi": ..., "event_tidak_sehat": ..., "total_data": ...}],
  "jam_puncak_per_kota": [{"kota": ..., "jam_puncak": ..., "sesi": ..., "avg_aqi_puncak": ...}],
  "generated_at": "ISO-8601 timestamp"
}
```

In [ ]:
# -------------------------------------------------------
# Konversi semua hasil analisis ke satu file JSON gabungan
# -------------------------------------------------------
import json
import os
from datetime import datetime, timezone, timedelta

wib = timezone(timedelta(hours=7))

# Rename agar konsisten dengan format yang disepakati A5
distribusi_records = (hasil_analisis
    .withColumnRenamed('kategori_baru', 'kategori')
    .toPandas()
    .to_dict('records'))

aqi_per_jam_records   = hasil2.toPandas().to_dict('records')
jam_puncak_records    = jam_puncak_labeled.toPandas().to_dict('records')
ranking_records       = ranking_kota.toPandas().to_dict('records')

results = {
    'distribusi_kategori':  distribusi_records,
    'aqi_per_jam':          aqi_per_jam_records,
    'ranking_kota':         ranking_records,
    'jam_puncak_per_kota':  jam_puncak_records,
    'generated_at':         datetime.now(wib).isoformat(),
}

# Path relatif terhadap repository root (notebook dijalankan dari folder spark/)
OUT_PATH = os.path.abspath(os.path.join('..', 'dashboard', 'data', 'spark_results.json'))
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2, default=str)

print(f'spark_results.json tersimpan ke: {OUT_PATH}')
print(f'   distribusi_kategori : {len(results["distribusi_kategori"])} baris')
print(f'   aqi_per_jam         : {len(results["aqi_per_jam"])} baris')
print(f'   ranking_kota        : {len(results["ranking_kota"])} baris')
print(f'   jam_puncak_per_kota : {len(results["jam_puncak_per_kota"])} baris')
print(f'   generated_at        : {results["generated_at"]}')

## AQI-B2 (Bonus +5): Spark MLlib — Prediksi AQI dengan Linear Regression

Membangun model regresi linier sederhana untuk memprediksi AQI berdasarkan **jam dalam sehari**
dan **identitas kota**. Output prediksi (per kota per jam) ditambahkan ke `spark_results.json`
dengan key `prediksi_aqi` agar dashboard bisa menampilkan kurva prediksi sebagai pelengkap data live.

> **Catatan**: jika jumlah event historis di HDFS masih sedikit (mis. < 100 baris), hasil model
> belum bisa diandalkan secara statistik — tetap kami latih sebagai demonstrasi pipeline ML.

In [ ]:
# -------------------------------------------------------
# Bonus: Linear Regression dengan Spark MLlib
# Feature: jam (0-23) + kota_encoded (StringIndexer + OneHot)
# Target : aqi
# -------------------------------------------------------
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

df_ml = (df_api
    .filter(col('aqi').isNotNull() & col('jam').isNotNull() & col('kota').isNotNull())
    .select('kota', 'jam', 'aqi'))

n_rows = df_ml.count()
print(f'Jumlah baris untuk training: {n_rows}')

indexer = StringIndexer(inputCol='kota', outputCol='kota_idx', handleInvalid='keep')
encoder = OneHotEncoder(inputCols=['kota_idx'], outputCols=['kota_vec'])
assembler = VectorAssembler(inputCols=['jam', 'kota_vec'], outputCol='features')
lr = LinearRegression(featuresCol='features', labelCol='aqi', regParam=0.1)

pipeline = Pipeline(stages=[indexer, encoder, assembler, lr])

In [ ]:
# Split 80:20 dan train
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f'Train rows: {train_df.count()} | Test rows: {test_df.count()}')

model = pipeline.fit(train_df)

# Evaluasi
predictions_test = model.transform(test_df)
evaluator_rmse = RegressionEvaluator(labelCol='aqi', predictionCol='prediction', metricName='rmse')
evaluator_r2   = RegressionEvaluator(labelCol='aqi', predictionCol='prediction', metricName='r2')

rmse = evaluator_rmse.evaluate(predictions_test)
r2   = evaluator_r2.evaluate(predictions_test)
print(f'RMSE: {rmse:.2f}')
print(f'R2  : {r2:.3f}')

predictions_test.select('kota', 'jam', 'aqi', 'prediction').show(10, truncate=False)

In [ ]:
# -------------------------------------------------------
# Hasilkan prediksi AQI untuk seluruh jam (0-23) di setiap kota
# -------------------------------------------------------
from pyspark.sql import Row

kota_list = [r['kota'] for r in df_ml.select('kota').distinct().collect()]
grid_rows = [Row(kota=k, jam=h, aqi=0.0) for k in kota_list for h in range(24)]
grid_df = spark.createDataFrame(grid_rows)

preds_full = (model.transform(grid_df)
    .select('kota', 'jam', spark_round('prediction', 1).alias('predicted_aqi'))
    .orderBy('kota', 'jam'))

print('=== PREDIKSI AQI PER JAM PER KOTA (sample) ===')
preds_full.show(15, truncate=False)

prediksi_records = preds_full.toPandas().to_dict('records')
print(f'Total prediksi rows: {len(prediksi_records)}')

In [ ]:
# -------------------------------------------------------
# Tambahkan hasil MLlib ke spark_results.json
# -------------------------------------------------------
results_with_ml = dict(results)
results_with_ml['prediksi_aqi'] = prediksi_records
results_with_ml['model_metrics'] = {
    'algorithm': 'LinearRegression',
    'features': ['jam', 'kota (one-hot)'],
    'train_rows': int(train_df.count()),
    'test_rows':  int(test_df.count()),
    'rmse':       round(float(rmse), 2),
    'r2':         round(float(r2), 3),
}
results_with_ml['generated_at'] = datetime.now(wib).isoformat()

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(results_with_ml, f, ensure_ascii=False, indent=2, default=str)

print(f'spark_results.json di-update dengan prediksi MLlib ({len(prediksi_records)} rows)')
print(f'   RMSE: {rmse:.2f}  |  R2: {r2:.3f}')

### Interpretasi Model

- **RMSE** memberitahu rata-rata error prediksi pada skala AQI. Bila RMSE sebanding atau lebih
  besar dari standar deviasi AQI di data, model belum lebih baik dari menebak rata-rata.
- **R²** menunjukkan proporsi variansi AQI yang berhasil dijelaskan. R² rendah (< 0.3) wajar untuk
  model dengan dua fitur saja — pola AQI nyata dipengaruhi cuaca, kelembapan, arah angin, jenis hari
  (kerja vs libur), yang belum kami masukkan.

**Langkah peningkatan ke depan**: tambahkan fitur cuaca (suhu, kelembapan, kecepatan angin),
indikator hari kerja vs akhir pekan, lag AQI 1-3 jam sebelumnya, dan ganti `LinearRegression`
dengan `GBTRegressor` atau `RandomForestRegressor` untuk menangkap pola non-linier.